# Part 3

In [1]:
from google.colab import userdata
from openai import OpenAI
import os

# 1. Retrieve the secret from Colab
openai_key = userdata.get('OpenAI_API')

# 2. Set it as an environment variable (optional, but helpful)
os.environ["OPENAI_API_KEY"] = openai_key

# 3. Initialize the client
client = OpenAI(api_key=openai_key)

# 4. Run your test
try:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": "Say: Working!"}],
        max_tokens=5
    )
    print(f"✓ Success! Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"✗ Connection Error: {e}")

!pip install langgraph langchain_openai

✓ Success! Response: Working!
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 23.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.12
    Uninstalling langchain-core-1.2.12:
      Successfully uninstalled langchain-core-1.2.12


In [2]:
"""
Manual Tool Calling Exercise
Students will see how tool calling works under the hood.
"""

import json
import math
from openai import OpenAI

# ============================================
# PART 1: Define Your Tools
# ============================================

def get_weather(location: str) -> str:
    """Get the current weather for a location"""
    # Simulated weather data
    weather_data = {
        "San Francisco": "Sunny, 72°F",
        "New York": "Cloudy, 55°F",
        "London": "Rainy, 48°F",
        "Tokyo": "Clear, 65°F"
    }
    return weather_data.get(location, f"Weather data not available for {location}")

def calculator(expression: str) -> str:
    """Evaluate a mathematical or geometric expression."""
    try:
        # Use a safe evaluation method like math functions combined with literal_eval
        # Geometric functions are available via the math module
        safe_dict = {
            "math": math,
            "sin": math.sin,
            "cos": math.cos,
            "tan": math.tan,
            "sqrt": math.sqrt,
            "pi": math.pi,
            "pow": pow,
            "radians": math.radians
        }
        # For this manual exercise, we use a simple eval with restricted globals
        # Instruction suggests using numexpr or ast.literal_eval for production
        result = eval(expression, {"__builtins__": None}, safe_dict)
        return json.dumps({"result": result})
    except Exception as e:
        return json.dumps({"error": str(e)})


# ============================================
# PART 2: Describe Tools to the LLM
# ============================================

# This is the JSON schema that tells the LLM what tools exist
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city name, e.g. San Francisco"
                    }
                },
                "required": ["location"]
            }
        }
    },
    # TODO: Students will add a second tool here (e.g., calculator)
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Perform calculations including geometric functions like sin, cos, and sqrt.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The math expression to evaluate, e.g., 'sin(radians(45)) * 10'"
                    }
                },
                "required": ["expression"]
            }
        }
    }
]


# ============================================
# PART 3: The Agent Loop
# ============================================

def run_agent(user_query: str):
    """
    Simple agent that can use tools.
    Shows the manual loop that LangGraph automates.
    """

    # Initialize OpenAI client
    client = OpenAI()

    # Start conversation with user query
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the provided tools when needed."},
        {"role": "user", "content": user_query}
    ]

    print(f"User: {user_query}\n")

    # Agent loop - can iterate up to 5 times
    for iteration in range(5):
        print(f"--- Iteration {iteration + 1} ---")

        # Call the LLM
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,  # ← This tells the LLM what tools are available
            tool_choice="auto"  # Let the model decide whether to use tools
        )

        assistant_message = response.choices[0].message

        # Check if the LLM wants to call a tool
        if assistant_message.tool_calls:
            print(f"LLM wants to call {len(assistant_message.tool_calls)} tool(s)")

            # Add the assistant's response to messages
            messages.append(assistant_message)

            # Execute each tool call
            for tool_call in assistant_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)

                print(f"  Tool: {function_name}")
                print(f"  Args: {function_args}")

                # THIS IS THE MANUAL DISPATCH
                # In a real system, you'd use a dictionary lookup
                if function_name == "get_weather":
                    result = get_weather(**function_args)
                elif function_name == "calculator":
                    result = calculator(**function_args)
                else:
                    result = f"Error: Unknown function {function_name}"

                print(f"  Result: {result}")

                # Add the tool result back to the conversation
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result
                })

            print()
            # Loop continues - LLM will see the tool results

        else:
            # No tool calls - LLM provided a final answer
            print(f"Assistant: {assistant_message.content}\n")
            return assistant_message.content

    return "Max iterations reached"


# ============================================
# PART 4: Test It
# ============================================

if __name__ == "__main__":
    # Test query that requires tool use
    print("="*60)
    print("TEST 1: Query requiring tool")
    print("="*60)
    run_agent("What's the weather like in San Francisco?")

    print("\n" + "="*60)
    print("TEST 2: Query not requiring tool")
    print("="*60)
    run_agent("Say hello!")

    print("\n" + "="*60)
    print("TEST 3: Multiple tool calls")
    print("="*60)
    run_agent("What's the weather in New York and London?")

    print("\n" + "="*60)
    print("TEST 4: Calculator tool call")
    print("="*60)
    run_agent("What is the area of a circle with a radius of 12.5, and what is the sine of that radius in radians?")


TEST 1: Query requiring tool
User: What's the weather like in San Francisco?

--- Iteration 1 ---
LLM wants to call 1 tool(s)
  Tool: get_weather
  Args: {'location': 'San Francisco'}
  Result: Sunny, 72°F

--- Iteration 2 ---
Assistant: The weather in San Francisco is sunny with a temperature of 72°F.


TEST 2: Query not requiring tool
User: Say hello!

--- Iteration 1 ---
Assistant: Hello! How can I assist you today?


TEST 3: Multiple tool calls
User: What's the weather in New York and London?

--- Iteration 1 ---
LLM wants to call 2 tool(s)
  Tool: get_weather
  Args: {'location': 'New York'}
  Result: Cloudy, 55°F
  Tool: get_weather
  Args: {'location': 'London'}
  Result: Rainy, 48°F

--- Iteration 2 ---
Assistant: The current weather is as follows:
- **New York**: Cloudy, 55°F
- **London**: Rainy, 48°F


TEST 4: Calculator tool call
User: What is the area of a circle with a radius of 12.5, and what is the sine of that radius in radians?

--- Iteration 1 ---
LLM wants to call 2 

# Part 4

In [5]:
"""
Manual Tool Calling Exercise
Students will see how tool calling works under the hood.
"""

import json
import math
from openai import OpenAI

# ============================================
# PART 1: Define Your Tools
# ============================================

def get_weather(location: str) -> str:
    """Get the current weather for a location"""
    # Simulated weather data
    weather_data = {
        "San Francisco": "Sunny, 72°F",
        "New York": "Cloudy, 55°F",
        "London": "Rainy, 48°F",
        "Tokyo": "Clear, 65°F"
    }
    return weather_data.get(location, f"Weather data not available for {location}")

# Tool 1: Calculator
def calculator(expression: str) -> str:
    """Evaluate a mathematical or geometric expression."""
    try:
        safe_dict = {"math": math, "sin": math.sin, "cos": math.cos, "sqrt": math.sqrt, "pi": math.pi, "radians": math.radians}
        result = eval(expression, {"__builtins__": None}, safe_dict)
        return json.dumps({"result": result})
    except Exception as e:
        return json.dumps({"error": str(e)})

# Tool 2: Letter Counter
def count_letters(text: str, letter: str) -> str:
    """Counts the number of occurrences of a specific letter in a piece of text."""
    count = text.lower().count(letter.lower())
    return json.dumps({"letter": letter, "count": count})

# Tool 3: Invention - Simple Currency Converter (USD to EUR @ 0.92)
def convert_currency(usd_amount: float) -> str:
    """Converts USD to EUR using a fixed rate of 0.92."""
    eur_amount = usd_amount * 0.92
    return json.dumps({"usd": usd_amount, "eur": round(eur_amount, 2)})


# ============================================
# PART 2: Describe Tools to the LLM
# ============================================

# This is the JSON schema that tells the LLM what tools exist
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city name, e.g. San Francisco"
                    }
                },
                "required": ["location"]
            }
        }
    },
    # TODO: Students will add a second tool here (e.g., calculator)
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Perform calculations including geometric functions like sin, cos, and sqrt.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The math expression to evaluate, e.g., 'sin(radians(45)) * 10'"
                    }
                },
                "required": ["expression"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "count_letters",
            "description": "Counts occurrences of a letter in a string",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {"type": "string"},
                    "letter": {"type": "string", "maxLength": 1}
                },
                "required": ["text", "letter"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": "Converts USD amount to EUR",
            "parameters": {
                "type": "object",
                "properties": {
                    "usd_amount": {"type": "number"}
                },
                "required": ["usd_amount"]
            }
        }
    }
]


# ============================================
# PART 3: The Agent Loop
# ============================================

def run_agent(user_query: str):
    """
    Simple agent that can use tools.
    Shows the manual loop that LangGraph automates.
    """

    # Initialize OpenAI client
    client = OpenAI()

    # Start conversation with user query
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the provided tools when needed."},
        {"role": "user", "content": user_query}
    ]

    print(f"User: {user_query}\n")

    # Build tool list
    tools_list = [get_weather, calculator, count_letters, convert_currency]
    tool_map = {tool.__name__: tool for tool in tools_list}

    # Agent loop - can iterate up to 5 times
    for iteration in range(5):
        print(f"--- Iteration {iteration + 1} ---")

        # Call the LLM
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,  # ← This tells the LLM what tools are available
            tool_choice="auto"  # Let the model decide whether to use tools
        )

        assistant_message = response.choices[0].message

        # Check if the LLM wants to call a tool
        if assistant_message.tool_calls:
            print(f"LLM wants to call {len(assistant_message.tool_calls)} tool(s)")

            # Add the assistant's response to messages
            messages.append(assistant_message)

            # Execute each tool call
            for tool_call in assistant_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)

                print(f"  Tool: {function_name}")
                print(f"  Args: {function_args}")

                if function_name in tool_map:
                    result = tool_map[function_name](**function_args)
                else:
                    result = f"Error: Unknown function {function_name}"

                print(f"  Result: {result}")

                # Add the tool result back to the conversation
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": result
                })

            print()
            # Loop continues - LLM will see the tool results

        else:
            # No tool calls - LLM provided a final answer
            print(f"Assistant: {assistant_message.content}\n")
            return assistant_message.content

    return "Max iterations reached"


# ============================================
# PART 4: Test It
# ============================================

if __name__ == "__main__":
    # Test query that requires tool use
    print("="*60)
    print("TEST 1: Query requiring tool")
    print("="*60)
    run_agent("What's the weather like in San Francisco?")

    print("\n" + "="*60)
    print("TEST 2: Query not requiring tool")
    print("="*60)
    run_agent("Say hello!")

    print("\n" + "="*60)
    print("TEST 3: Multiple tool calls (Parallel)")
    print("="*60)
    run_agent("What's the weather in New York and London?")

    print("\n" + "="*60)
    print("TEST 4: Multiple tool use (Same Turn)")
    print("="*60)
    run_agent("Are there more i's than s's in Mississippi riverboats?")

    print("\n" + "="*60)
    print("TEST 5: Sequential Chaining")
    print("="*60)
    run_agent("What is the sin of the difference between the number of i's and the number of s's in Mississippi riverboats?")

    print("\n" + "="*60)
    print("TEST 6: All Tools & 5-Turn Limit")
    print("="*60)
    run_agent("Count 'm' in 'mammoth', find the square root of that 'm' count, convert that USD to EUR, get the weather in Tokyo, and finally multiply that Tokyo temperature by the EUR amount.")


TEST 1: Query requiring tool
User: What's the weather like in San Francisco?

--- Iteration 1 ---
LLM wants to call 1 tool(s)
  Tool: get_weather
  Args: {'location': 'San Francisco'}
  Result: Sunny, 72°F

--- Iteration 2 ---
Assistant: The weather in San Francisco is currently sunny with a temperature of 72°F.


TEST 2: Query not requiring tool
User: Say hello!

--- Iteration 1 ---
Assistant: Hello! How can I assist you today?


TEST 3: Multiple tool calls (Parallel)
User: What's the weather in New York and London?

--- Iteration 1 ---
LLM wants to call 2 tool(s)
  Tool: get_weather
  Args: {'location': 'New York'}
  Result: Cloudy, 55°F
  Tool: get_weather
  Args: {'location': 'London'}
  Result: Rainy, 48°F

--- Iteration 2 ---
Assistant: The weather is as follows:

- **New York**: Cloudy, 55°F
- **London**: Rainy, 48°F


TEST 4: Multiple tool use (Same Turn)
User: Are there more i's than s's in Mississippi riverboats?

--- Iteration 1 ---
LLM wants to call 2 tool(s)
  Tool: count_

# Part 5

In [4]:
import json
import math
import sqlite3
from typing import Annotated, TypedDict
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, ToolMessage
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# --- PART 1: Define Tools (Keep your existing functions) ---
# [get_weather, calculator, count_letters, convert_currency functions here]

# --- PART 2: Setup LangGraph State ---
class State(TypedDict):
    # add_messages ensures new messages are appended to the history
    messages: Annotated[list[BaseMessage], add_messages]

# --- PART 3: Define Graph Nodes ---
# Initialize the model and bind tools
llm = ChatOpenAI(model="gpt-4o-mini").bind_tools(tools)

def call_model(state: State):
    """Node to call the LLM."""
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def tool_node(state: State):
    """Node to execute tools based on the last LLM message."""
    last_message = state["messages"][-1]
    tool_outputs = []

    # tool_map from your code
    tools_list = [get_weather, calculator, count_letters, convert_currency]
    tool_map = {tool.__name__: tool for tool in tools_list}

    for tool_call in last_message.tool_calls:
        fn_name = tool_call["name"]
        args = tool_call["args"]

        if fn_name in tool_map:
            result = tool_map[fn_name](**args)
        else:
            result = f"Error: {fn_name} not found."

        tool_outputs.append(ToolMessage(
            tool_call_id=tool_call["id"],
            content=str(result)
        ))
    return {"messages": tool_outputs}

def should_continue(state: State):
    """Conditional edge to check if more tool calls are needed."""
    last_message = state["messages"][-1]
    if not last_message.tool_calls:
        return END
    return "tools"

# --- PART 4: Build the Graph ---
workflow = StateGraph(State)

workflow.add_node("agent", call_model)
workflow.add_node("tools", tool_node)

workflow.set_entry_point("agent")
workflow.add_conditional_edges("agent", should_continue)
workflow.add_edge("tools", "agent")

# Enable Checkpointing for recovery and conversation context
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

# --- PART 5: Run Conversation ---
config = {"configurable": {"thread_id": "1"}} # thread_id enables context recovery

def chat(query: str):
    print(f"\nUser: {query}")
    for event in app.stream({"messages": [HumanMessage(content=query)]}, config):
        for value in event.values():
            last_msg = value["messages"][-1]
            if hasattr(last_msg, "content") and last_msg.content:
                if not isinstance(last_msg, ToolMessage):
                    print(f"Assistant: {last_msg.content}")

if __name__ == "__main__":
    # This now runs as a single long conversation
    chat("My name is Jacob.")
    chat("What is the square root of 144?")
    chat("Remember my name? Also, what's the weather in Tokyo?")


User: My name is Jacob.
Assistant: Nice to meet you, Jacob! How can I assist you today?

User: What is the square root of 144?
Assistant: The square root of 144 is 12.

User: Remember my name? Also, what's the weather in Tokyo?
Assistant: Yes, your name is Jacob! 

The weather in Tokyo is clear with a temperature of 65°F.
